# Full Road Classification + Speed Decision Pipeline
Combines 4 models into one final speed decision per image:
1. **Your trained classifier**  autoroute (highway) / urbaine (urban) / rurale (rural)
2. **Roboflow surface model**  off-road (unpaved) override
3. **YOLOv11 damage model (Hugging Face)**  damaged vs. not
4. **YOLOv11 traffic sign model (GitHub)**  speed limits, stop signs, traffic lights


## 1. Install dependencies

In [ ]:
!pip install roboflow ultralytics huggingface_hub --quiet


## 2. PASTE YOUR FINAL TRAINED CLASSIFIER HERE
Everything that defines `model`, `CLASS_NAMES`, `IMG_SIZE` from your other notebook (imports, config, sequence labels, dataset build, training). Paste as separate cells, same order as your original notebook.

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
CLASS_NAMES = ["autoroute", "urbaine", "rurale"]  

MODEL_PATH = "/kaggle/input/models/khawlaelhamdi/road-classifier/keras/rural-urban-highway/1/road_classifier (2).keras"

model = tf.keras.models.load_model(MODEL_PATH)
print("Loaded trained classifier from:", MODEL_PATH)
model.summary()

## 2b. Load your trained speed bump detector (from trainv2)
Same idea as the classifier above — point this at the Kaggle Model you uploaded from trainv2's output.

*** UNVERIFIED: update `SPEED_BUMP_MODEL_PATH` to your actual Kaggle Model path/version. ***

In [ ]:
from ultralytics import YOLO

SPEED_BUMP_MODEL_PATH = "/kaggle/input/models/khawlaelhamdi/bumpybumps/pytorch/1/1/speed_bump_detector.pt"  # UNVERIFIED: fix to your actual upload path

bump_model = YOLO(SPEED_BUMP_MODEL_PATH)
print("Loaded trained speed bump detector from:", SPEED_BUMP_MODEL_PATH)
print("Classes:", bump_model.names)

## 2c. Load your trained pedestrian + crosswalk detector (from trainv3)
Same idea again — point this at the Kaggle Model you uploaded from trainv3's output.

*** UNVERIFIED: update `PED_CROSSWALK_MODEL_PATH` to your actual Kaggle Model path/version. ***

In [ ]:
PED_CROSSWALK_MODEL_PATH = "/kaggle/input/models/khawlaelhamdi/crosswalks-and-ppl/pytorch/1/1/ped_crosswalk_detector.pt"  
ped_crosswalk_model = YOLO(PED_CROSSWALK_MODEL_PATH)
print("Loaded trained pedestrian + crosswalk detector from:", PED_CROSSWALK_MODEL_PATH)
print("Classes:", ped_crosswalk_model.names)

## 2d. Load your trained weather classifier (from trainv4)
Exported as `.tflite` (not `.keras` like the road classifier), so it loads through
`tf.lite.Interpreter` instead of `tf.keras.models.load_model`.

*** UNVERIFIED: update `WEATHER_MODEL_PATH` to your actual Kaggle Model path/version. ***

In [ ]:
WEATHER_MODEL_PATH = "/kaggle/input/models/khawlaelhamdi/weather-classifier/tensorflow2/default/1/weather_classifier.tflite"  

WEATHER_CLASSES = ["clear", "adverse"]  # must match TARGET_CLASSES order from trainv4

weather_interpreter = tf.lite.Interpreter(model_path=WEATHER_MODEL_PATH)
weather_interpreter.allocate_tensors()
weather_input_details = weather_interpreter.get_input_details()
weather_output_details = weather_interpreter.get_output_details()
print("Loaded trained weather classifier from:", WEATHER_MODEL_PATH)

## 2e. Load the vehicle detector for safety distance
Stock COCO-pretrained YOLOv8n — no custom training, no Kaggle Model upload needed, it already knows
`car`/`truck`/`bus`.

In [ ]:
vehicle_model = YOLO("yolov8n.pt")
print("Loaded stock YOLOv8n for vehicle detection. Classes:", vehicle_model.names)

## 3. Off-road (unpaved) detection — Roboflow surface model
*** UNVERIFIED: `VERSION` number and exact "unpaved" class string — run 3a first. ***

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "gGXHUPHkw6vMLDGV21DF" 
ROBOFLOW_WORKSPACE = "sri-lab"
ROBOFLOW_PROJECT = "road-surface-classification-lgxl1"
ROBOFLOW_VERSION = 1 

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
offroad_project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
offroad_model = offroad_project.version(ROBOFLOW_VERSION).model
print("Roboflow off-road model loaded.")


### 3a. Discover the real class names

In [ ]:
UNPAVED_CLASS_NAMES = {"Unpaved"}
OFFROAD_CONFIDENCE_THRESHOLD = 0.5

def detect_offroad(image_path):
    result = offroad_model.predict(image_path).json()
    predictions = result.get("predictions", [])
    if not predictions:
        return False, 0.0
    top = predictions[0] if isinstance(predictions, list) else predictions
    predicted_class = top.get("class") or top.get("top", "")
    confidence = top.get("confidence", 0.0)
    is_offroad = predicted_class in UNPAVED_CLASS_NAMES and confidence >= OFFROAD_CONFIDENCE_THRESHOLD
    return is_offroad, confidence

## 4. Damage detection — YOLOv11 (Hugging Face)
*** UNVERIFIED: exact weights filename — run the discovery cell first. ***

In [ ]:
damage_project = rf.workspace("roaddamage-msfnj").project("road-damage-ww8ex")
damage_model_rf = damage_project.version(1).model  

print("Roboflow damage model loaded.")

In [ ]:
DAMAGE_CONFIDENCE_THRESHOLD = 0.5

def detect_damage(image_path):
    result = damage_model_rf.predict(image_path, confidence=int(DAMAGE_CONFIDENCE_THRESHOLD * 100), overlap=30).json()
    detections = result.get("predictions", [])
    
    parsed = [{"class": d.get("class"), "confidence": d.get("confidence", 0.0)} for d in detections]
    max_confidence = max((d["confidence"] for d in parsed), default=0.0)
    is_damaged = len(parsed) > 0

    return is_damaged, max_confidence, parsed

## 5. Traffic sign detection — YOLOv11 (GitHub, bhaskrr/traffic-sign-detection-using-yolov11)
Confirmed 15 classes: Green Light, Red Light, Speed Limit 20/30/40/50/60/70/80/90/100/110/120, Stop (+ a bogus "all" class from the source dataset export, excluded below).

*** UNVERIFIED: exact .pt filename inside the repo's `model/` folder — run the discovery cell first. ***

In [ ]:
!git clone https://github.com/bhaskrr/traffic-sign-detection-using-yolov11.git /kaggle/working/traffic_sign_repo


In [ ]:
import os

model_dir = "/kaggle/working/traffic_sign_repo/model"
print("Files in the model/ folder:")
for f in os.listdir(model_dir):
    print(" ", f)


In [ ]:
# 1. Install ultralytics (run this if it's not already installed in your environment)
!pip install ultralytics -q

# 2. Import the YOLO class along with os
import os
from ultralytics import YOLO

# 3. Your existing code will now work perfectly
!git clone https://github.com/bhaskrr/traffic-sign-detection-using-yolov11.git /kaggle/working/traffic_sign_repo

model_dir = "/kaggle/working/traffic_sign_repo/model"
print("Files in the model/ folder:")
for f in os.listdir(model_dir):
    print(" ", f)
    
SIGN_WEIGHTS_FILENAME = "traffic_sign_detector.pt"  
sign_weights_path = os.path.join(model_dir, SIGN_WEIGHTS_FILENAME)

sign_model = YOLO(sign_weights_path)
print("Traffic sign model loaded. Classes:", sign_model.names)


In [ ]:
SIGN_CONFIDENCE_THRESHOLD = 0.5
IGNORED_SIGN_CLASSES = {"all"} 

def detect_traffic_signs(image_path):
    """
    Returns a list of detected signs above threshold, e.g.:
      [{"class": "Speed Limit 60", "confidence": 0.91}, {"class": "Stop", "confidence": 0.87}]
    """
    results = sign_model.predict(image_path, verbose=False)
    detections = []
    for r in results:
        for box in r.boxes:
            conf = float(box.conf[0])
            cls_name = sign_model.names[int(box.cls[0])]
            if cls_name in IGNORED_SIGN_CLASSES or conf < SIGN_CONFIDENCE_THRESHOLD:
                continue
            detections.append({"class": cls_name, "confidence": conf})
    return detections


## 5b. Speed bump detection

In [ ]:
SPEED_BUMP_CONFIDENCE_THRESHOLD = 0.5

def detect_speed_bump(image_path):
    """
    Returns (bump_ahead: bool, max_confidence: float, detections: list).
    Only the 'Speed-Bump' class triggers a slowdown — 'Rumble Strip' is detected and returned for
    visibility but doesn't currently change speed on its own.
    """
    results = bump_model.predict(image_path, conf=SPEED_BUMP_CONFIDENCE_THRESHOLD, verbose=False)
    detections = []
    for r in results:
        for box in r.boxes:
            conf = float(box.conf[0])
            cls_name = bump_model.names[int(box.cls[0])]
            detections.append({"class": cls_name, "confidence": conf})

    bump_detections = [d for d in detections if d["class"] == "Speed-Bump"]
    bump_ahead = len(bump_detections) > 0
    max_confidence = max((d["confidence"] for d in bump_detections), default=0.0)

    return bump_ahead, max_confidence, detections

## 5c. Pedestrian + crosswalk detection
*** NOTE: "pedestrian ahead" here is currently just "a person was confidently detected anywhere in
frame" — it doesn't yet check whether they're actually in the car's path or how far away they are.
Good enough for an MVP hard-stop trigger, but worth refining later (e.g. only trigger if the box is
large/centered/near the bottom of frame). ***

In [ ]:
PEDESTRIAN_CONFIDENCE_THRESHOLD = 0.5
CROSSWALK_CONFIDENCE_THRESHOLD = 0.5

def detect_pedestrian_crosswalk(image_path):
    """
    Returns (pedestrian_ahead: bool, crosswalk_ahead: bool, detections: list).
    """
    results = ped_crosswalk_model.predict(image_path, conf=min(PEDESTRIAN_CONFIDENCE_THRESHOLD, CROSSWALK_CONFIDENCE_THRESHOLD), verbose=False)
    detections = []
    for r in results:
        for box in r.boxes:
            conf = float(box.conf[0])
            cls_name = ped_crosswalk_model.names[int(box.cls[0])]
            detections.append({"class": cls_name, "confidence": conf})

    pedestrian_detections = [d for d in detections if d["class"] == "person" and d["confidence"] >= PEDESTRIAN_CONFIDENCE_THRESHOLD]
    crosswalk_detections = [d for d in detections if d["class"] == "cross walk" and d["confidence"] >= CROSSWALK_CONFIDENCE_THRESHOLD]

    pedestrian_ahead = len(pedestrian_detections) > 0
    crosswalk_ahead = len(crosswalk_detections) > 0

    return pedestrian_ahead, crosswalk_ahead, detections

## 5d. Weather detection

In [ ]:
def detect_weather(image_path):
    """
    Returns (weather_condition: str, confidence: float) — one of WEATHER_CLASSES.
    """
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0).astype(weather_input_details[0]["dtype"])

    weather_interpreter.set_tensor(weather_input_details[0]["index"], img_array)
    weather_interpreter.invoke()
    predictions = weather_interpreter.get_tensor(weather_output_details[0]["index"])[0]

    weather_condition = WEATHER_CLASSES[np.argmax(predictions)]
    confidence = float(np.max(predictions))

    return weather_condition, confidence

## 5e. Safety distance detection
Heuristic only, not a real metric distance — a vehicle whose box is tall relative to the frame, or
whose bottom edge sits low in the frame, is considered close. Thresholds below are placeholders from
`safety_distance_calibration.ipynb` — replace with whatever you calibrated on your own sample frames.

*** UNVERIFIED: confirm `CLOSE_RELATIVE_HEIGHT_THRESHOLD` / `CLOSE_RELATIVE_BOTTOM_THRESHOLD` against your
own calibration results before trusting this. ***

In [ ]:
VEHICLE_CLASSES = ["car", "truck", "bus"]
VEHICLE_CONFIDENCE_THRESHOLD = 0.4
CLOSE_RELATIVE_HEIGHT_THRESHOLD = 0.35  # UNVERIFIED placeholder — calibrate against your own frames
CLOSE_RELATIVE_BOTTOM_THRESHOLD = 0.85   # UNVERIFIED placeholder — calibrate against your own frames

def detect_car_too_close(image_path):
    """
    Returns (too_close: bool, closest_vehicle: dict or None, detections: list).
    """
    img = Image.open(image_path)
    frame_w, frame_h = img.size

    results = vehicle_model.predict(image_path, conf=VEHICLE_CONFIDENCE_THRESHOLD, verbose=False)
    detections = []

    for r in results:
        for box in r.boxes:
            cls_name = vehicle_model.names[int(box.cls[0])]
            if cls_name not in VEHICLE_CLASSES:
                continue
            conf = float(box.conf[0])
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            relative_height = (y2 - y1) / frame_h
            relative_bottom = y2 / frame_h

            detections.append({
                "class": cls_name,
                "confidence": conf,
                "relative_height": relative_height,
                "relative_bottom": relative_bottom,
            })

    close_vehicles = [
        d for d in detections
        if d["relative_height"] >= CLOSE_RELATIVE_HEIGHT_THRESHOLD
        or d["relative_bottom"] >= CLOSE_RELATIVE_BOTTOM_THRESHOLD
    ]

    too_close = len(close_vehicles) > 0
    closest = max(close_vehicles, key=lambda d: d["relative_bottom"], default=None)

    return too_close, closest, detections

## 6. Base speed table — road type + damage
Placeholder values — confirm/tune the actual numbers with your supervisor.

In [ ]:
BASE_SPEED_BY_ROAD_TYPE = {
    "autoroute": 120,
    "urbaine": 60,
    "rurale": 100,
    "off-road": 50,
}

DAMAGE_SPEED_REDUCTION = 0.30  # 30% reduction when damaged

def get_base_speed(road_type, is_damaged):
    speed = BASE_SPEED_BY_ROAD_TYPE[road_type]
    if is_damaged:
        speed = speed * (1 - DAMAGE_SPEED_REDUCTION)
    return round(speed, 1)

## 7. Speed override logic (pedestrian, signs, crosswalk, speed bump, safety distance, weather)
Priority, highest to lowest:
1. **Pedestrian detected** — forces speed to 0.
2. **Stop / Red light** — forces speed to 0, no exceptions.
3. **Crosswalk ahead** — caps speed to `CROSSWALK_SPEED_CAP`.
4. **Speed bump ahead** — caps speed to `SPEED_BUMP_SPEED_CAP`.
5. **Posted speed limit sign** — caps the cruise speed (never raises it above the base spec).
6. **Car too close ahead** — soft reduction (`SAFETY_DISTANCE_SPEED_REDUCTION`), advisory rather than a
   hard cap — it's a "back off" suggestion, not a hazard on the level of a pedestrian or stop sign.
7. **Adverse weather** — soft reduction (`WEATHER_SPEED_REDUCTION`), same advisory logic as safety
   distance, applied last since it's the softest signal in the chain.


In [ ]:
import re

CROSSWALK_SPEED_CAP = 30           # km/h — placeholder, confirm the actual number with your supervisor
SPEED_BUMP_SPEED_CAP = 30          # km/h — placeholder, confirm the actual number with your supervisor
SAFETY_DISTANCE_SPEED_REDUCTION = 0.20  # 20% reduction when a vehicle ahead is too close — placeholder
WEATHER_SPEED_REDUCTION = 0.20          # 20% reduction in adverse weather — placeholder

def apply_overrides(
    current_speed,
    detected_signs,
    pedestrian_ahead=False,
    crosswalk_ahead=False,
    bump_ahead=False,
    car_too_close=False,
    weather_condition="clear",
):
    # Highest priority: pedestrian detected -> full stop
    if pedestrian_ahead:
        return 0, "Pedestrian detected — stop"

    # Next: stop / red light -> full stop, no exceptions
    for sign in detected_signs:
        if sign["class"] in ("Stop", "Red Light"):
            return 0, f"{sign['class']} detected — mandatory stop"

    # NOTE: Yellow/amber light is NOT a class this model detects — can't apply
    # the "slow by 50%" rule until a model that covers this class is added.

    final_speed = current_speed
    override_reason = None

    # Crosswalk — caps speed, doesn't touch it if already below the cap
    if crosswalk_ahead and CROSSWALK_SPEED_CAP < final_speed:
        final_speed = CROSSWALK_SPEED_CAP
        override_reason = f"Crosswalk detected — capped to {CROSSWALK_SPEED_CAP} km/h"

    # Speed bump — caps speed, doesn't touch it if already below the cap
    if bump_ahead and SPEED_BUMP_SPEED_CAP < final_speed:
        final_speed = SPEED_BUMP_SPEED_CAP
        override_reason = f"Speed bump detected — capped to {SPEED_BUMP_SPEED_CAP} km/h"

    # Speed limit signs cap the current speed (never raise it)
    for sign in detected_signs:
        match = re.search(r"Speed Limit (\d+)", sign["class"])
        if match:
            posted_limit = int(match.group(1))
            if posted_limit < final_speed:
                final_speed = posted_limit
                override_reason = f"Capped to posted limit: {posted_limit} km/h"

    # Safety distance — soft, advisory reduction (not a hard cap)
    if car_too_close:
        final_speed = round(final_speed * (1 - SAFETY_DISTANCE_SPEED_REDUCTION), 1)
        override_reason = "Vehicle too close ahead — reduce speed and increase following distance"

    # Weather — soft, advisory reduction (not a hard cap)
    if weather_condition == "adverse":
        final_speed = round(final_speed * (1 - WEATHER_SPEED_REDUCTION), 1)
        override_reason = "Adverse weather conditions — reduce speed"

    return final_speed, override_reason

## 8. Full combined pipeline

In [ ]:
def classify_full(image_path):
    """
    Runs all models on one image and returns the complete decision:
      road_type, damaged, detected_signs, pedestrian_ahead, crosswalk_ahead, speed_bump_detected,
      car_too_close, weather_condition, base_speed, final_speed_kmh, override_reason
    """
    # Road type: off-road check first, overrides the 3-class model if confidently unpaved
    is_offroad, offroad_conf = detect_offroad(image_path)
    if is_offroad:
        road_type = "off-road"
    else:
        img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
        img_array = tf.keras.utils.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0)
        predictions = model.predict(img_array, verbose=0)
        road_type = CLASS_NAMES[np.argmax(predictions[0])]

    # Damage — independent of road type
    is_damaged, damage_conf, damage_detections = detect_damage(image_path)

    # Traffic signs
    detected_signs = detect_traffic_signs(image_path)

    # Speed bump — independent of everything else
    bump_ahead, bump_conf, bump_detections = detect_speed_bump(image_path)

    # Pedestrian + crosswalk — independent of everything else
    pedestrian_ahead, crosswalk_ahead, ped_detections = detect_pedestrian_crosswalk(image_path)

    # Safety distance — independent of everything else
    car_too_close, closest_vehicle, vehicle_detections = detect_car_too_close(image_path)

    # Weather — independent of everything else
    weather_condition, weather_conf = detect_weather(image_path)

    # Combine into final speed decision
    base_speed = get_base_speed(road_type, is_damaged)
    final_speed, override_reason = apply_overrides(
        base_speed,
        detected_signs,
        pedestrian_ahead=pedestrian_ahead,
        crosswalk_ahead=crosswalk_ahead,
        bump_ahead=bump_ahead,
        car_too_close=car_too_close,
        weather_condition=weather_condition,
    )

    return {
        "road_type": road_type,
        "damaged": is_damaged,
        "detected_signs": detected_signs,
        "pedestrian_ahead": pedestrian_ahead,
        "crosswalk_ahead": crosswalk_ahead,
        "speed_bump_detected": bump_ahead,
        "car_too_close": car_too_close,
        "weather_condition": weather_condition,
        "base_speed": base_speed,
        "final_speed_kmh": final_speed,
        "override_reason": override_reason,
    }

## 9. Test on a real image

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np 

# Define the directory path instead of a single file
TEST_DIR = "/kaggle/input/datasets/khawlaelhamdi/testtest"

# Check if the directory exists before starting the loop
if os.path.exists(TEST_DIR):
    # Loop through all files in the test directory
    for f in os.listdir(TEST_DIR):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            image_path = os.path.join(TEST_DIR, f)
            
            # 1. Run the classification function
            result = classify_full(image_path)
            
            # 2. Display the image
            img = Image.open(image_path)
            plt.figure()  # Creates a new figure for each image so they don't overwrite each other
            plt.imshow(img)
            plt.axis("off")
            
            # 3. Build and set the title
            title = f"File: {f}\n{result['road_type']} | damaged: {result['damaged']} | ped: {result['pedestrian_ahead']} | crosswalk: {result['crosswalk_ahead']} | bump: {result['speed_bump_detected']} | close: {result['car_too_close']} | weather: {result['weather_condition']} | speed: {result['final_speed_kmh']} km/h"
            if result["override_reason"]:
                title += f"\n({result['override_reason']})"
            plt.title(title)
            plt.show()
            
            # 4. Print raw result dictionary
            print(f"Results for {f}:")
            print(result)
            print("-" * 50)  # Visual separator in the console logs
else:
    print(f"Error: The directory {TEST_DIR} does not exist. Please check the dataset path.")

In [ ]:
model.save("/kaggle/working/road_classifier.keras")

In [ ]:
import os
import datetime

path = "/kaggle/working/road_classifier.keras"
mod_time = os.path.getmtime(path)
print("Last modified:", datetime.datetime.fromtimestamp(mod_time))
print("Current time: ", datetime.datetime.now())


# Voice Assistant


In [ ]:
!pip install openai-whisper gTTS --quiet

import re
import whisper


In [ ]:

# ---------------------------------------------------------------------------
# 1. Speech-to-text
# ---------------------------------------------------------------------------
 
whisper_model = whisper.load_model("base")  # "small" or "medium" if base isn't accurate enough
 
 
def transcribe_audio(audio_path, language="fr"):
    """Whisper: audio file -> plain text string. Nothing more."""
    result = whisper_model.transcribe(audio_path, language=language)
    return result["text"].strip()
 

In [ ]:

# ---------------------------------------------------------------------------
# 2. Intent parsing (rule-based — matches the approach we discussed earlier:
#    small, fixed vocabulary, keyword/regex matching is legitimate for this,
#    not a placeholder for something fancier)
# ---------------------------------------------------------------------------
 
# Each entry: (regex pattern, intent name, whether it needs a numeric value)
INTENT_PATTERNS = [
    (r"ferme.*fen[êe]tre|close.*window",                    "close_window",           False),
    (r"ouvre.*fen[êe]tre|open.*window",                      "open_window",            False),
    (r"limite.*vitesse|limit.*speed",                        "set_speed_limit",        True),
    (r"r[ée]gle.*vitesse.*croisi[èe]re|set.*cruise.*speed",  "set_cruise_speed",       True),
    (r"mode [ée]co|eco mode",                                 "enable_eco_mode",        False),
    (r"augmente.*distance|increase.*distance",                "increase_safety_distance", False),
    (r"diminue.*distance|reduis.*distance|decrease.*distance","decrease_safety_distance", False),
    (r"baisse.*vitesse|ralentis|lower.*speed|slow down",      "decrease_speed",         False),
    (r"augmente.*vitesse|acc[ée]l[èe]re|speed up",            "increase_speed",         False),
]
 
 
def extract_number(text):
    match = re.search(r"(\d+)", text)
    return int(match.group(1)) if match else None
 
 
def parse_intent(text):
    """
    Returns a dict: {"intent": ..., "value": ... or None, "raw_text": text}
    "intent" is "unknown" if nothing matched — this is a real, expected outcome,
    not an error, and the dispatcher/response logic both handle it explicitly.
    """
    text_lower = text.lower()
 
    for pattern, intent, needs_number in INTENT_PATTERNS:
        if re.search(pattern, text_lower):
            value = extract_number(text_lower) if needs_number else None
            return {"intent": intent, "value": value, "raw_text": text}
 
    return {"intent": "unknown", "value": None, "raw_text": text}
 
 

In [ ]:

 
# ---------------------------------------------------------------------------
# 3. Command dispatcher — updates a simple in-memory vehicle state.
#    Swap the body of this function for real CAN/simulated-CAN calls later —
#    everything upstream (parsing, TTS) doesn't need to change when you do.
# ---------------------------------------------------------------------------
 
def dispatch_command(parsed_intent, vehicle_state):
    intent = parsed_intent["intent"]
    value = parsed_intent["value"]
 
    if intent == "set_speed_limit" and value is not None:
        vehicle_state["speed_limit_kmh"] = value
    elif intent == "set_cruise_speed" and value is not None:
        vehicle_state["cruise_speed_kmh"] = value
    elif intent == "enable_eco_mode":
        vehicle_state["eco_mode"] = True
    elif intent == "increase_safety_distance":
        vehicle_state["safety_distance_level"] += 1
    elif intent == "decrease_safety_distance":
        vehicle_state["safety_distance_level"] = max(1, vehicle_state["safety_distance_level"] - 1)
    elif intent == "decrease_speed":
        vehicle_state["cruise_speed_kmh"] = max(0, vehicle_state["cruise_speed_kmh"] - 10)
    elif intent == "increase_speed":
        vehicle_state["cruise_speed_kmh"] += 10
    elif intent == "close_window":
        vehicle_state["window_open"] = False
    elif intent == "open_window":
        vehicle_state["window_open"] = True
    # "unknown" intent -> vehicle_state untouched on purpose
 
    return vehicle_state
 
 

In [ ]:

# ---------------------------------------------------------------------------
# 4. Response text — what the assistant actually says back
# ---------------------------------------------------------------------------
 
RESPONSE_TEMPLATES = {
    "close_window":              "D'accord, je ferme la fenêtre.",
    "open_window":                "D'accord, j'ouvre la fenêtre.",
    "set_speed_limit":            "Limite de vitesse réglée à {value} kilomètres heure.",
    "set_cruise_speed":           "Vitesse de croisière réglée à {value} kilomètres heure.",
    "enable_eco_mode":            "Mode éco activé.",
    "increase_safety_distance":   "J'augmente la distance de sécurité.",
    "decrease_safety_distance":   "Je réduis la distance de sécurité.",
    "decrease_speed":             "D'accord, je ralentis.",
    "increase_speed":             "D'accord, j'accélère.",
    "unknown":                    "Désolé, je n'ai pas compris cette commande.",
}
 
 
def generate_response_text(parsed_intent):
    template = RESPONSE_TEMPLATES[parsed_intent["intent"]]
    if "{value}" in template and parsed_intent["value"] is not None:
        return template.format(value=parsed_intent["value"])
    return template

In [ ]:
from gtts import gTTS

def speak(text, output_path="response.mp3", lang="fr"):
    tts = gTTS(text=text, lang=lang)
    tts.save(output_path)
    return output_path
 

In [ ]:

# ---------------------------------------------------------------------------
# 6. Full voice command pipeline — ties 1-5 together
# ---------------------------------------------------------------------------
 
def process_voice_command(audio_path, vehicle_state, language="fr"):
    """
    Full loop: audio in -> vehicle state updated + spoken response generated.
    Returns (updated_vehicle_state, response_text, response_audio_path)
    """
    text = transcribe_audio(audio_path, language=language)
    parsed = parse_intent(text)
    vehicle_state = dispatch_command(parsed, vehicle_state)
 
    response_text = generate_response_text(parsed)
    response_audio_path = speak(response_text)
 
    return vehicle_state, response_text, response_audio_path
 

In [ ]:

# ---------------------------------------------------------------------------
# 7. Proactive speed-suggestion announcements (from the image classifier)
#    Only speaks when the suggested speed actually CHANGES — this is the
#    "only informs the driver if the speed changes" requirement.
# ---------------------------------------------------------------------------
 
SPEED_SUGGESTION_TEMPLATES = {
    "autoroute": "Autoroute détectée. Vitesse suggérée : {speed} kilomètres heure.",
    "urbaine":   "Zone urbaine détectée. Vitesse suggérée : {speed} kilomètres heure.",
    "rurale":    "Route rurale détectée. Vitesse suggérée : {speed} kilomètres heure.",
    "off-road":  "Terrain hors route détecté. Vitesse suggérée : {speed} kilomètres heure.",
}
 
_last_announced_speed = {"value": None}  # module-level state, tracks the last spoken speed
 
 
def announce_speed_suggestion(road_type, suggested_speed, override_reason=None):
    """
    Call this after classify_full() runs on a new frame.
    Speaks only if suggested_speed differs from the last announced value.
    Returns the spoken text, or None if it stayed silent (no change).
    """
    if suggested_speed == _last_announced_speed["value"]:
        return None  # no change since last time -> stay silent
 
    _last_announced_speed["value"] = suggested_speed
 
    if override_reason:
        # A traffic sign or stop/red light forced this change — say why
        text = f"{override_reason}. Vitesse suggérée : {suggested_speed} kilomètres heure."
    else:
        template = SPEED_SUGGESTION_TEMPLATES.get(
            road_type, "Vitesse suggérée : {speed} kilomètres heure."
        )
        text = template.format(speed=suggested_speed)
 
    speak(text, output_path="speed_announcement.mp3")
    return text
 
 
def reset_speed_announcement_tracking():
    """Call this if you need to force the next announcement to speak regardless of value (e.g. on startup)."""
    _last_announced_speed["value"] = None
 
 

In [ ]:
# ---------------------------------------------------------------------------
# 8. Demo / manual testing — no microphone needed for this part
# ---------------------------------------------------------------------------

import nest_asyncio
nest_asyncio.apply()  # lets asyncio.run() work inside Jupyter's own event loop

if __name__ == "__main__":
    vehicle_state = {
        "cruise_speed_kmh": 50,
        "speed_limit_kmh": None,
        "eco_mode": False,
        "safety_distance_level": 2,
        "window_open": True,
    }

    # --- Test intent parsing directly with text, no audio file needed ---
    test_commands = [
        "Règle la vitesse de croisière à 120 km/h",
        "Limite la vitesse à 60",
        "Active le mode éco",
        "Ferme la fenêtre",
        "Blablabla nonsense command",
    ]

    print("--- Testing intent parsing (text only, no audio) ---")
    for cmd in test_commands:
        parsed = parse_intent(cmd)
        vehicle_state = dispatch_command(parsed, vehicle_state)
        response_text = generate_response_text(parsed)
        print(f"Command: {cmd!r}")
        print(f"  -> Parsed: {parsed}")
        print(f"  -> Response: {response_text}")
    print("\nFinal vehicle state:", vehicle_state)

    # --- Test the speed-announcement change-detection logic ---
    print("\n--- Testing speed suggestion announcements ---")
    print(announce_speed_suggestion("autoroute", 120))   # should speak (first time)
    print(announce_speed_suggestion("autoroute", 120))   # should stay silent (no change)
    print(announce_speed_suggestion("urbaine", 60))       # should speak (changed)
    print(announce_speed_suggestion("urbaine", 60, override_reason="Stop détecté"))  # speaks (reason attached)

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import Audio, display

TEST_DIR = "/kaggle/input/datasets/khawlaelhamdi/testtest"

if os.path.exists(TEST_DIR):
    for f in os.listdir(TEST_DIR):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            image_path = os.path.join(TEST_DIR, f)

            # 1. Run the classification function
            result = classify_full(image_path)

            # 2. Display the image
            img = Image.open(image_path)
            plt.figure()
            plt.imshow(img)
            plt.axis("off")

            # 3. Build and set the title
            title = f"File: {f}\n{result['road_type']} | damaged: {result['damaged']} | ped: {result['pedestrian_ahead']} | crosswalk: {result['crosswalk_ahead']} | bump: {result['speed_bump_detected']} | close: {result['car_too_close']} | weather: {result['weather_condition']} | speed: {result['final_speed_kmh']} km/h"
            if result["override_reason"]:
                title += f"\n({result['override_reason']})"
            plt.title(title)
            plt.show()

            # 4. Print raw result dictionary
            print(f"Results for {f}:")
            print(result)

            # 5. Voice assistant: only speaks if the suggested speed actually changed
            spoken_text = announce_speed_suggestion(
                result["road_type"],
                result["final_speed_kmh"],
                override_reason=result["override_reason"],
            )

            if spoken_text:
                print(f"🔊 Assistant says: {spoken_text}")
                display(Audio("speed_announcement.mp3", autoplay=False))
            else:
                print("🔇 Assistant stays silent (speed unchanged)")

            print("-" * 50)
else:
    print(f"Error: The directory {TEST_DIR} does not exist. Please check the dataset path.")